# Parte 4 — Integração Multimodal: Late Fusion (EfficientNetB1 + MLP de Metadados)

## Motivação

O dataset HAM10000 não é composto apenas por pixels — ele inclui **metadados clínicos vitais**:
- **Idade** do paciente
- **Sexo** biológico
- **Localização anatômica** da lesão

Ignorar esses dados é um erro comum. Estudos mostram que a inclusão de metadados pode aumentar a acurácia em **3–5%** ao fornecer contexto clínico fundamental — por exemplo, a probabilidade de certas lesões malignas varia significativamente com a idade e a localização corporal.

## Arquitetura: Late Fusion

```
Imagem (600×450) ──► EfficientNetB1 ──► Flatten ──► Dense(256) ──┐
                                                                    ├──► Concatenate ──► Dense(128) ──► Softmax(7)
Metadados (idade, sexo, localização) ──► MLP(64→32) ────────────┘
```

As características extraídas de ambos os ramos são **concatenadas antes da camada Softmax final** (Late Fusion), permitindo que o modelo pese automaticamente a contribuição relativa da imagem e dos metadados.

## Codificação dos Metadados

| Variável | Estratégia | Tratamento de Nulos |
|----------|-----------|--------------------|
| Idade | Normalização numérica (min-max) | Imputação pela mediana |
| Sexo | One-hot encoding | Categoria "unknown" |
| Localização | One-hot encoding (15 sítios) | Categoria "unknown" |

In [ ]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.preprocessing import label_binarize

# Add project root to path
sys.path.insert(0, os.path.abspath(".."))
from utils.utils_preproc import create_dl_splits, format_center_crop_tf
from utils.utils_model import get_callbacks, plot_history, BatchTimeCallback

print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {keras.__version__}")
print(f"GPUs disponíveis: {tf.config.list_physical_devices('GPU')}")

## 1. Carregamento e Preparação dos Dados

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR    = "../data"
META_PATH   = os.path.join(DATA_DIR, "HAM10000_metadata")
IMG_DIRS    = [
    os.path.join(DATA_DIR, "HAM10000_images_part_1"),
    os.path.join(DATA_DIR, "HAM10000_images_part_2"),
]
LABEL_PATH  = "../label2idx.json"
CKPT_PATH   = "../checkpoints/model_d_multimodal_best.weights.h5"

# ── Load metadata ─────────────────────────────────────────────────────────────
df = pd.read_csv(META_PATH)
print(f"Total de imagens: {len(df)}")
df.head(3)

In [ ]:
# ── Resolve image paths ───────────────────────────────────────────────────────
img_map = {}
for d in IMG_DIRS:
    for f in os.listdir(d):
        if f.endswith(".jpg"):
            img_id = f.replace(".jpg", "")
            img_map[img_id] = os.path.join(d, f)

df["image_path"] = df["image_id"].map(img_map)
missing = df["image_path"].isna().sum()
print(f"Imagens encontradas: {len(df) - missing}/{len(df)}")
df = df.dropna(subset=["image_path"]).reset_index(drop=True)

# ── Label encoding ────────────────────────────────────────────────────────────
if os.path.exists(LABEL_PATH):
    with open(LABEL_PATH) as f:
        label2idx = json.load(f)
else:
    classes = sorted(df["dx"].unique())
    label2idx = {c: i for i, c in enumerate(classes)}
    with open(LABEL_PATH, "w") as f:
        json.dump(label2idx, f)

idx2label = {v: k for k, v in label2idx.items()}
df["dx_encoded"] = df["dx"].map(label2idx)

print(f"\nClasses ({len(label2idx)}): {label2idx}")

## 2. Pré-processamento dos Metadados

In [ ]:
# ── Age: impute median, then normalize ────────────────────────────────────────
age_median = df["age"].median()
df["age_clean"] = df["age"].fillna(age_median)

scaler = MinMaxScaler()
df["age_norm"] = scaler.fit_transform(df[["age_clean"]])

# ── Sex: one-hot (male, female, unknown) ─────────────────────────────────────
sex_dummies = pd.get_dummies(df["sex"], prefix="sex")
# Ensure all expected columns exist
for col in ["sex_male", "sex_female", "sex_unknown"]:
    if col not in sex_dummies.columns:
        sex_dummies[col] = 0
sex_dummies = sex_dummies[["sex_male", "sex_female", "sex_unknown"]]

# ── Localization: one-hot ────────────────────────────────────────────────────
loc_dummies = pd.get_dummies(df["localization"], prefix="loc")

# ── Concatenate all metadata features ────────────────────────────────────────
meta_df = pd.concat([df[["age_norm"]], sex_dummies, loc_dummies], axis=1)
META_DIM = meta_df.shape[1]
print(f"Dimensão do vetor de metadados: {META_DIM} features")
print(f"  - 1 feature numérica (idade normalizada)")
print(f"  - 3 features de sexo (one-hot)")
print(f"  - {loc_dummies.shape[1]} features de localização (one-hot)")

meta_matrix = meta_df.values.astype(np.float32)
df["meta_idx"] = range(len(df))  # index para recuperar vetor de metadados

In [ ]:
# ── Visualização da distribuição de metadados ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Distribuição dos Metadados Clínicos", fontsize=14)

# Age distribution
axes[0].hist(df["age_clean"], bins=20, color="#7F77DD", edgecolor="white")
axes[0].set_title("Distribuição de Idade")
axes[0].set_xlabel("Idade (anos)")
axes[0].set_ylabel("Contagem")

# Sex distribution
sex_counts = df["sex"].value_counts()
axes[1].bar(sex_counts.index, sex_counts.values, color=["#4E79A7", "#F28E2B", "#E15759"])
axes[1].set_title("Distribuição de Sexo")
axes[1].set_ylabel("Contagem")

# Top localizations
loc_counts = df["localization"].value_counts().head(8)
axes[2].barh(loc_counts.index, loc_counts.values, color="#59A14F")
axes[2].set_title("Localizações Anatômicas (Top 8)")
axes[2].set_xlabel("Contagem")

plt.tight_layout()
plt.show()

In [ ]:
# ── Correlação metadados × diagnóstico ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Metadados por Diagnóstico", fontsize=14)

# Age by diagnosis
classes_order = sorted(df["dx"].unique())
age_by_dx = [df[df["dx"] == c]["age_clean"].values for c in classes_order]
axes[0].boxplot(age_by_dx, labels=classes_order)
axes[0].set_title("Faixa Etária por Diagnóstico")
axes[0].set_xlabel("Diagnóstico")
axes[0].set_ylabel("Idade")
axes[0].tick_params(axis='x', rotation=30)

# Sex distribution per diagnosis
sex_dx = df.groupby(["dx", "sex"]).size().unstack(fill_value=0)
sex_dx.plot(kind="bar", ax=axes[1], stacked=True, 
            color=["#4E79A7", "#F28E2B", "#E15759"])
axes[1].set_title("Distribuição de Sexo por Diagnóstico")
axes[1].set_xlabel("Diagnóstico")
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title="Sexo")

plt.tight_layout()
plt.show()

## 3. Divisão Train/Val/Test

In [ ]:
train_df, val_df, test_df = create_dl_splits(df, train_frac=0.8, val_frac=0.1, test_frac=0.1)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

## 4. Pipeline de Dados Multimodal (Imagem + Metadados)

In [ ]:
BATCH_SIZE = 32
IMG_SIZE   = (224, 224)

# Pre-compute metadata tensors for each split
train_meta = meta_matrix[train_df["meta_idx"].values]
val_meta   = meta_matrix[val_df["meta_idx"].values]
test_meta  = meta_matrix[test_df["meta_idx"].values]


def load_image_and_meta(img_path, meta_vec, label):
    """Load & preprocess a single dermoscopy image alongside its metadata vector."""
    img = tf.io.read_file(img_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.cast(img, tf.float32)
    img = format_center_crop_tf(img)              # center crop → 224×224
    img = tf.keras.applications.efficientnet.preprocess_input(img)
    return (img, meta_vec), label


def make_multimodal_dataset(df_split, meta_arr, shuffle=False, repeat=False):
    """Build a tf.data pipeline that yields ((image, metadata), label) tuples."""
    paths  = df_split["image_path"].values
    labels = df_split["dx_encoded"].values.astype(np.int32)

    ds = tf.data.Dataset.from_tensor_slices((paths, meta_arr, labels))
    ds = ds.map(load_image_and_meta, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(buffer_size=min(2000, len(df_split)), seed=42)
    if repeat:
        ds = ds.repeat()

    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = make_multimodal_dataset(train_df, train_meta, shuffle=True,  repeat=True)
val_ds   = make_multimodal_dataset(val_df,   val_meta,   shuffle=False, repeat=False)
test_ds  = make_multimodal_dataset(test_df,  test_meta,  shuffle=False, repeat=False)

# Sanity check
(imgs_batch, meta_batch), labels_batch = next(iter(train_ds))
print(f"Imagens: {imgs_batch.shape}")
print(f"Metadados: {meta_batch.shape}")
print(f"Labels: {labels_batch.shape}")

## 5. Construção do Modelo Late Fusion

```
┌─────────────────────────────────────────────────────────┐
│  Ramo de Imagem (CNN)                                    │
│  Input (224,224,3) → EfficientNetB1 → GAP → Dense(256)  │
└─────────────────────────────────────────────────────────┘
                          │
                    Concatenate  ← ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ┐
                          │                                   │
                     Dense(128)                               │
                       Dropout                    ┌────────────────────────┐
                     Softmax(7)                   │  Ramo de Metadados     │
                                                  │  Input(META_DIM)        │
                                                  │  Dense(64) → Dense(32) │
                                                  └────────────────────────┘
```

In [ ]:
N_CLASSES = len(label2idx)


def build_late_fusion_model(meta_dim, n_classes=7, freeze_backbone=True):
    """
    Constrói um modelo de Late Fusion com dois ramos:
      - Ramo de Imagem: EfficientNetB1 (pré-treinada no ImageNet)
      - Ramo de Metadados: MLP pequena
    
    Os dois ramos são concatenados antes da camada de classificação final.
    
    Parameters
    ----------
    meta_dim     : int   — Dimensão do vetor de metadados
    n_classes    : int   — Número de classes de saída
    freeze_backbone : bool — Se True, congela o backbone na Fase 1
    
    Returns
    -------
    keras.Model com duas entradas: (img_input, meta_input)
    """
    # ── Ramo de Imagem ──────────────────────────────────────────────────────
    img_input = keras.Input(shape=(224, 224, 3), name="image_input")

    backbone = keras.applications.EfficientNetB1(
        include_top=False,
        weights="imagenet",
        input_tensor=img_input,
    )
    backbone.trainable = not freeze_backbone

    x = backbone.output
    x = keras.layers.GlobalAveragePooling2D(name="gap")(x)
    x = keras.layers.Dense(256, activation="relu", name="img_dense")(x)
    x = keras.layers.Dropout(0.4, name="img_dropout")(x)
    img_features = x  # shape: (batch, 256)

    # ── Ramo de Metadados (MLP) ─────────────────────────────────────────────
    meta_input = keras.Input(shape=(meta_dim,), name="meta_input")

    m = keras.layers.Dense(64, activation="relu", name="meta_dense1")(meta_input)
    m = keras.layers.BatchNormalization(name="meta_bn1")(m)
    m = keras.layers.Dropout(0.3, name="meta_dropout1")(m)
    m = keras.layers.Dense(32, activation="relu", name="meta_dense2")(m)
    meta_features = m  # shape: (batch, 32)

    # ── Integração Late Fusion ──────────────────────────────────────────────
    combined = keras.layers.Concatenate(name="late_fusion")([img_features, meta_features])

    out = keras.layers.Dense(128, activation="relu", name="fusion_dense")(combined)
    out = keras.layers.Dropout(0.3, name="fusion_dropout")(out)
    out = keras.layers.Dense(n_classes, name="logits")(out)  # raw logits

    model = keras.Model(
        inputs=[img_input, meta_input],
        outputs=out,
        name="late_fusion_model",
    )
    return model


model = build_late_fusion_model(META_DIM, N_CLASSES, freeze_backbone=True)
model.summary(show_trainable=True)

## 6. Treinamento — Fase 1: Backbone Congelado (Head only)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Class weights (to address imbalance: mel, df, vasc are rare)
classes = np.array(sorted(df["dx"].unique()))
cw_vals = compute_class_weight(class_weight="balanced", classes=classes, y=df["dx"])
class_weights = {i: w for i, w in enumerate(cw_vals)}

print("Class weights:")
for i, (cls, w) in enumerate(zip(classes, cw_vals)):
    print(f"  {i} ({cls}): {w:.3f}")

In [ ]:
STEPS_PER_EPOCH = len(train_df) // BATCH_SIZE

# ── Fase 1: treinar apenas o head + ramo de metadados ─────────────────────────
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

callbacks_ph1 = get_callbacks(
    checkpoint_path=CKPT_PATH.replace(".h5", "_phase1.h5"),
    patience_es=8,
    patience_lr=4,
)

print("=== Fase 1: Backbone congelado (apenas head + metadados) ===")
history_ph1 = model.fit(
    train_ds,
    steps_per_epoch=STEPS_PER_EPOCH,
    epochs=20,
    validation_data=val_ds,
    callbacks=callbacks_ph1,
    class_weight=class_weights,
    verbose=1,
)

plot_history(history_ph1, "Late Fusion — Fase 1 (Backbone Congelado)")

## 7. Treinamento — Fase 2: Fine-tuning do Backbone

In [ ]:
# ── Fase 2: descongelar o backbone inteiro com lr muito baixo ─────────────────
backbone_layer = model.get_layer("efficientnetb1")
backbone_layer.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),  # lr 100× menor
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)

trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"Parâmetros treináveis na Fase 2: {trainable_params:,}")

callbacks_ph2 = get_callbacks(
    checkpoint_path=CKPT_PATH,
    patience_es=10,
    patience_lr=4,
)

print("=== Fase 2: Fine-tuning do backbone EfficientNetB1 ===")
history_ph2 = model.fit(
    train_ds,
    steps_per_epoch=STEPS_PER_EPOCH,
    epochs=30,
    validation_data=val_ds,
    callbacks=callbacks_ph2,
    class_weight=class_weights,
    verbose=1,
)

plot_history(history_ph2, "Late Fusion — Fase 2 (Fine-tuning)")

## 8. Avaliação no Conjunto de Teste

In [ ]:
# ── Carregar os melhores pesos salvos ─────────────────────────────────────────
if os.path.exists(CKPT_PATH):
    model.load_weights(CKPT_PATH)
    print("Melhores pesos carregados.")

# ── Coletar predições ─────────────────────────────────────────────────────────
y_true, y_pred_proba = [], []

for (imgs, metas), labels in test_ds:
    logits = model([imgs, metas], training=False)
    proba  = tf.nn.softmax(logits).numpy()
    y_pred_proba.append(proba)
    y_true.extend(labels.numpy())

y_pred_proba = np.vstack(y_pred_proba)
y_true       = np.array(y_true)
y_pred       = np.argmax(y_pred_proba, axis=1)
class_names  = [idx2label[i] for i in range(N_CLASSES)]

print(f"Amostras no conjunto de teste: {len(y_true)}")

In [ ]:
# ── Classification report ─────────────────────────────────────────────────────
print("=" * 65)
print("  Modelo D — Late Fusion (EfficientNetB1 + Metadados)")
print("=" * 65)
print(classification_report(y_true, y_pred, target_names=class_names))

mel_idx    = label2idx["mel"]
mel_recall = (y_pred[y_true == mel_idx] == mel_idx).mean()
print(f"  *** Recall de Melanoma: {mel_recall:.3f} ***  (métrica clínica prioritária)")

In [ ]:
# ── Matriz de confusão ────────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=class_names, yticklabels=class_names,
)
plt.title("Late Fusion — Matriz de Confusão (Teste)")
plt.ylabel("Label Real")
plt.xlabel("Label Previsto")
plt.tight_layout()
plt.show()

In [ ]:
# ── AUC-ROC Macro ─────────────────────────────────────────────────────────────
y_bin     = label_binarize(y_true, classes=list(range(N_CLASSES)))
macro_auc = roc_auc_score(y_bin, y_pred_proba, average="macro", multi_class="ovr")
print(f"  Macro AUC-ROC: {macro_auc:.4f}")

# ── Curvas ROC por classe ─────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
for i, cls in enumerate(class_names):
    fpr, tpr, _ = roc_curve(y_bin[:, i], y_pred_proba[:, i])
    auc_score   = roc_auc_score(y_bin[:, i], y_pred_proba[:, i])
    lw = 2.5 if cls == "mel" else 1.2
    ax.plot(fpr, tpr, lw=lw, label=f"{cls} (AUC={auc_score:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=0.8)
ax.set_xlabel("Taxa de Falso Positivo")
ax.set_ylabel("Taxa de Verdadeiro Positivo")
ax.set_title("Late Fusion — Curvas ROC por Classe  (melanoma em negrito)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## 9. Análise do Ganho dos Metadados

Para quantificar a contribuição dos metadados, treinamos uma versão **ablation** do modelo usando apenas a imagem (zerando os metadados) e comparamos com o modelo completo.

In [ ]:
# ── Ablation: predição com metadados zerados ──────────────────────────────────
y_pred_noMeta_proba = []
zero_meta = np.zeros((BATCH_SIZE, META_DIM), dtype=np.float32)

for (imgs, metas), labels in test_ds:
    batch_size_actual = imgs.shape[0]
    empty_meta = tf.zeros((batch_size_actual, META_DIM), dtype=tf.float32)
    logits = model([imgs, empty_meta], training=False)
    proba  = tf.nn.softmax(logits).numpy()
    y_pred_noMeta_proba.append(proba)

y_pred_noMeta_proba = np.vstack(y_pred_noMeta_proba)
y_pred_noMeta       = np.argmax(y_pred_noMeta_proba, axis=1)

macro_auc_noMeta = roc_auc_score(y_bin, y_pred_noMeta_proba, average="macro", multi_class="ovr")
mel_recall_noMeta = (y_pred_noMeta[y_true == mel_idx] == mel_idx).mean()

print("=" * 55)
print("  Impacto dos Metadados (Ablation Study)")
print("=" * 55)
print(f"  Modelo Completo (imagem + metadados):")
print(f"    Macro AUC-ROC : {macro_auc:.4f}")
print(f"    Recall Melanoma: {mel_recall:.3f}")
print()
print(f"  Ablation (apenas imagem, metadados=0):")
print(f"    Macro AUC-ROC : {macro_auc_noMeta:.4f}")
print(f"    Recall Melanoma: {mel_recall_noMeta:.3f}")
print()
print(f"  ΔMacro AUC-ROC  : {macro_auc - macro_auc_noMeta:+.4f}")
print(f"  ΔRecall Melanoma: {mel_recall - mel_recall_noMeta:+.3f}")

In [ ]:
# ── Gráfico comparativo ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Ablation Study: Impacto dos Metadados Clínicos", fontsize=14)

labels_plot = ["Imagem + Metadados", "Apenas Imagem (Ablation)"]
colors = ["#7F77DD", "#CCCCCC"]

axes[0].bar(labels_plot, [macro_auc, macro_auc_noMeta], color=colors)
axes[0].set_ylim(0.8, 1.0)
axes[0].set_title("Macro AUC-ROC")
axes[0].set_ylabel("AUC")
for i, v in enumerate([macro_auc, macro_auc_noMeta]):
    axes[0].text(i, v + 0.002, f"{v:.4f}", ha="center", fontsize=11)

axes[1].bar(labels_plot, [mel_recall, mel_recall_noMeta], color=colors)
axes[1].axhline(0.8, color="red", linestyle="--", lw=1.2, label="Alvo: 0.80")
axes[1].set_ylim(0, 1)
axes[1].set_title("Recall de Melanoma")
axes[1].set_ylabel("Recall")
axes[1].legend()
for i, v in enumerate([mel_recall, mel_recall_noMeta]):
    axes[1].text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=11)

plt.tight_layout()
plt.show()

## 10. Comparação Global dos Modelos

In [ ]:
# ── Consolidação dos resultados de todos os modelos ───────────────────────────
# Preencher com os resultados obtidos nos notebooks anteriores
all_results = {
    "A — CNN Custom":             {"macro_auc": 0.000, "mel_recall": 0.000},  # substituir
    "B — EfficientNetB0":         {"macro_auc": 0.000, "mel_recall": 0.000},  # substituir
    "C — MobileNetV2 / DenseNet": {"macro_auc": 0.000, "mel_recall": 0.000},  # substituir
    "D — Late Fusion (Este)": {
        "macro_auc":  round(macro_auc, 4),
        "mel_recall": round(mel_recall, 3),
    },
}

results_df = pd.DataFrame(all_results).T.reset_index()
results_df.columns = ["Modelo", "Macro AUC", "Recall Melanoma"]
print(results_df.to_string(index=False))

In [ ]:
# ── Gráfico comparativo final ─────────────────────────────────────────────────
colors = ["#BBBBBB", "#BBBBBB", "#BBBBBB", "#7F77DD"]  # destaque: modelo D

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Comparação Final de Todos os Modelos — Conjunto de Teste", fontsize=14)

axes[0].bar(results_df["Modelo"], results_df["Macro AUC"], color=colors)
axes[0].set_ylim(0, 1)
axes[0].set_title("Macro AUC-ROC")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(results_df["Modelo"], results_df["Recall Melanoma"], color=colors)
axes[1].axhline(0.8, color="red", linestyle="--", lw=1.2, label="Alvo: 0.80")
axes[1].set_ylim(0, 1)
axes[1].set_title("Recall de Melanoma  (métrica prioritária)")
axes[1].legend()
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

## 11. Conclusão

### O que foi construído

Um modelo de **Late Fusion multimodal** que combina:
- **Ramo de Imagem**: EfficientNetB1 pré-treinada no ImageNet como backbone, com camadas densas de projeção (256 unidades)
- **Ramo de Metadados**: MLP compacta (64→32 unidades) processando idade (normalizada), sexo (one-hot) e localização anatômica (one-hot)
- **Fusão**: Concatenação das representações de ambos os ramos, seguida de classificação final

### Pontos-chave

1. **Codificação Inteligente**: Idade normalizada via min-max; sexo e localização via one-hot com categoria "unknown" explícita para não perder amostras com dados faltantes

2. **Treinamento em 2 Fases**: Fase 1 com backbone congelado (treina apenas o head e o ramo de metadados); Fase 2 com fine-tuning completo em lr muito reduzido (1e-5)

3. **Ablation Study**: Comparação com metadados zerados quantifica o ganho real da integração clínica

4. **Motivação Clínica**: Metadados como localização e idade fornecem contexto que a imagem isolada não captura — ex. melanomas acrais têm prognóstico diferente dos tronco